# 01 – Exploration of the HR extracts

TechNova Partners faces unusually high attrition, and the HR team wants to understand the potential causes of resignations. In this notebook we inspect, clean and join the three HR extracts, then explore how leavers differ from stayers.

## 0. Setup

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update(
    {
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.grid.axis": "y",
        "axes.axisbelow": True,
        "grid.color": "#e5e5e5",
        "grid.linewidth": 0.8,
        "legend.frameon": False,
        "axes.titlesize": 13,
        "axes.titleweight": "bold",
        "axes.labelsize": 11,
        "figure.figsize": (8, 4.5),
    }
)
pd.set_option("display.max_columns", None)

RAW_DIR = Path("../data/raw")

sirh = pd.read_csv(RAW_DIR / "extrait_sirh.csv")
evaluations = pd.read_csv(RAW_DIR / "extrait_eval.csv")
survey = pd.read_csv(RAW_DIR / "extrait_sondage.csv")

In [2]:
def describe_source(df):
    """Display the shape, the first rows and the numeric and text summaries of a DataFrame."""
    print(df.shape)
    display(df.head(3))
    display(df.describe().T)
    display(df.describe(exclude="number").T)


def to_employee_id(series):
    """Convert an employee key to an integer by dropping the E_ prefix if present."""
    return series.astype(str).str.removeprefix("E_").astype(int)

## 1. The three source files

### 1.1 HR information system (`sirh`)

The brief describes this file as the role, age, salary, seniority and socio-demographic information of each employee.

In [3]:
describe_source(sirh)

(1470, 12)


,id_employee,age,genre,revenu_mensuel,statut_marital,departement,poste,nombre_experiences_precedentes,nombre_heures_travailless,annee_experience_totale,annees_dans_l_entreprise,annees_dans_le_poste_actuel
0,1,41,F,5993,Célibataire,Commercial,Cadre Commercial,8,80,8,6,4
1,2,49,M,5130,Marié(e),Consulting,Assistant de Direction,1,80,10,10,7
2,4,37,M,2090,Célibataire,Consulting,Consultant,6,80,7,0,0


,count,mean,std,min,25%,50%,75%,max
id_employee,1470.0,1024.865306,602.024335,1.0,491.25,1020.5,1555.75,2068.0
age,1470.0,36.923810,9.135373,18.0,30.00,36.0,43.00,60.0
revenu_mensuel,1470.0,6502.931293,4707.956783,1009.0,2911.00,4919.0,8379.00,19999.0
nombre_experiences_precedentes,1470.0,2.693197,2.498009,0.0,1.00,2.0,4.00,9.0
nombre_heures_travailless,1470.0,80.000000,0.000000,80.0,80.00,80.0,80.00,80.0
annee_experience_totale,1470.0,11.279592,7.780782,0.0,6.00,10.0,15.00,40.0
annees_dans_l_entreprise,1470.0,7.008163,6.126525,0.0,3.00,5.0,9.00,40.0
annees_dans_le_poste_actuel,1470.0,4.229252,3.623137,0.0,2.00,3.0,7.00,18.0


,count,unique,top,freq
genre,1470,2,M,882
statut_marital,1470,3,Marié(e),673
departement,1470,3,Consulting,961
poste,1470,9,Cadre Commercial,326


The file matches the brief, with one row per employee identified by `id_employee`. It covers the job title and department, age, gender, marital status and monthly income. It also holds three durations (total experience, seniority in the company, seniority in the current role) and the number of previous employers. A working-hours column seems constant.

### 1.2 Performance reviews (`evaluations`)

The brief describes this file as the annual performance ratings and the satisfaction scores given by employees.

In [4]:
describe_source(evaluations)

(1470, 10)


,satisfaction_employee_environnement,note_evaluation_precedente,niveau_hierarchique_poste,satisfaction_employee_nature_travail,satisfaction_employee_equipe,satisfaction_employee_equilibre_pro_perso,eval_number,note_evaluation_actuelle,heure_supplementaires,augementation_salaire_precedente
0,2,3,2,4,1,1,E_1,3,Oui,11 %
1,3,2,2,2,4,3,E_2,4,Non,23 %
2,4,2,1,3,2,3,E_4,3,Oui,15 %


,count,mean,std,min,25%,50%,75%,max
satisfaction_employee_environnement,1470.0,2.721769,1.093082,1.0,2.0,3.0,4.0,4.0
note_evaluation_precedente,1470.0,2.729932,0.711561,1.0,2.0,3.0,3.0,4.0
niveau_hierarchique_poste,1470.0,2.063946,1.106940,1.0,1.0,2.0,3.0,5.0
satisfaction_employee_nature_travail,1470.0,2.728571,1.102846,1.0,2.0,3.0,4.0,4.0
satisfaction_employee_equipe,1470.0,2.712245,1.081209,1.0,2.0,3.0,4.0,4.0
satisfaction_employee_equilibre_pro_perso,1470.0,2.761224,0.706476,1.0,2.0,3.0,3.0,4.0
note_evaluation_actuelle,1470.0,3.153741,0.360824,3.0,3.0,3.0,3.0,4.0


,count,unique,top,freq
eval_number,1470,1470,E_1,1
heure_supplementaires,1470,2,Non,1054
augementation_salaire_precedente,1470,15,11 %,210


The file holds more than the brief announces, with one row per employee identified by `eval_number`. Besides the previous and current performance ratings and the four satisfaction scores, it includes the hierarchical level, whether the employee works overtime, and the last salary raise, stored as text.

### 1.3 Well-being survey (`survey`)

The brief describes this file as the latest annual well-being survey, to which HR added a flag showing whether the employee left.

In [5]:
describe_source(survey)

(1470, 12)


,a_quitte_l_entreprise,nombre_participation_pee,nb_formations_suivies,nombre_employee_sous_responsabilite,code_sondage,distance_domicile_travail,niveau_education,domaine_etude,ayant_enfants,frequence_deplacement,annees_depuis_la_derniere_promotion,annes_sous_responsable_actuel
0,Oui,0,0,1,1,1,2,Infra & Cloud,Y,Occasionnel,0,5
1,Non,1,3,1,2,8,1,Infra & Cloud,Y,Frequent,1,7
2,Oui,0,3,1,4,2,2,Autre,Y,Occasionnel,0,0


,count,mean,std,min,25%,50%,75%,max
nombre_participation_pee,1470.0,0.793878,0.852077,0.0,0.00,1.0,1.00,3.0
nb_formations_suivies,1470.0,2.799320,1.289271,0.0,2.00,3.0,3.00,6.0
nombre_employee_sous_responsabilite,1470.0,1.000000,0.000000,1.0,1.00,1.0,1.00,1.0
code_sondage,1470.0,1024.865306,602.024335,1.0,491.25,1020.5,1555.75,2068.0
distance_domicile_travail,1470.0,9.192517,8.106864,1.0,2.00,7.0,14.00,29.0
niveau_education,1470.0,2.912925,1.024165,1.0,2.00,3.0,4.00,5.0
annees_depuis_la_derniere_promotion,1470.0,2.187755,3.222430,0.0,0.00,1.0,3.00,15.0
annes_sous_responsable_actuel,1470.0,4.123129,3.568136,0.0,2.00,3.0,7.00,17.0


,count,unique,top,freq
a_quitte_l_entreprise,1470,2,Non,1233
domaine_etude,1470,6,Infra & Cloud,606
ayant_enfants,1470,1,Y,1470
frequence_deplacement,1470,3,Occasionnel,1043


The file holds facts rather than opinions, with one row per employee identified by `code_sondage`. It covers training, participation in the employee savings plan (PEE), commute distance, education level and field, business travel, years since the last promotion and years under the current manager. It also holds the attrition flag and two columns that seem constant.

## 2. Variable inventory

Pandas only sees how each column is stored, so we tag every column with its kind. The kind will decide which chart and which encoding fit each variable.

In [6]:
sources = {"sirh": sirh, "evaluations": evaluations, "survey": survey}

COLUMN_KINDS = {
    "id_employee": "key",
    "eval_number": "key",
    "code_sondage": "key",
    "a_quitte_l_entreprise": "target",
    "nombre_heures_travailless": "constant",
    "nombre_employee_sous_responsabilite": "constant",
    "ayant_enfants": "constant",
    "genre": "binary",
    "heure_supplementaires": "binary",
    "statut_marital": "nominal",
    "departement": "nominal",
    "poste": "nominal",
    "domaine_etude": "nominal",
    "satisfaction_employee_environnement": "ordinal",
    "satisfaction_employee_nature_travail": "ordinal",
    "satisfaction_employee_equipe": "ordinal",
    "satisfaction_employee_equilibre_pro_perso": "ordinal",
    "note_evaluation_precedente": "ordinal",
    "note_evaluation_actuelle": "ordinal",
    "niveau_hierarchique_poste": "ordinal",
    "niveau_education": "ordinal",
    "frequence_deplacement": "ordinal",
    "age": "quantitative",
    "revenu_mensuel": "quantitative",
    "augementation_salaire_precedente": "quantitative",
    "nombre_experiences_precedentes": "quantitative",
    "nb_formations_suivies": "quantitative",
    "nombre_participation_pee": "quantitative",
    "distance_domicile_travail": "quantitative",
    "annee_experience_totale": "quantitative",
    "annees_dans_l_entreprise": "quantitative",
    "annees_dans_le_poste_actuel": "quantitative",
    "annees_depuis_la_derniere_promotion": "quantitative",
    "annes_sous_responsable_actuel": "quantitative",
}

rows = []
for source_name, df in sources.items():
    for col in df.columns:
        examples = df[col].dropna().unique()[:2]
        rows.append(
            {
                "source": source_name,
                "column": col,
                "dtype": str(df[col].dtype),
                "n_unique": df[col].nunique(),
                "examples": ", ".join(map(str, examples)),
            }
        )

inventory = pd.DataFrame(rows)
inventory.insert(3, "kind", inventory["column"].map(COLUMN_KINDS))
assert inventory["kind"].notna().all(), inventory.loc[
    inventory["kind"].isna(), "column"
].tolist()

display(inventory)
inventory["kind"].value_counts().to_frame("n_columns")

,source,column,dtype,kind,n_unique,examples
0,sirh,id_employee,int64,key,1470,"1, 2"
1,sirh,age,int64,quantitative,43,"41, 49"
2,sirh,genre,str,binary,2,"F, M"
3,sirh,revenu_mensuel,int64,quantitative,1349,"5993, 5130"
4,sirh,statut_marital,str,nominal,3,"Célibataire, Marié(e)"
5,sirh,departement,str,nominal,3,"Commercial, Consulting"
6,sirh,poste,str,nominal,9,"Cadre Commercial, Assistant de Direction"
7,sirh,nombre_experiences_precedentes,int64,quantitative,10,"8, 1"
8,sirh,nombre_heures_travailless,int64,constant,1,80
9,sirh,annee_experience_totale,int64,quantitative,40,"8, 10"


,n_columns
kind,
quantitative,12
ordinal,9
nominal,4
key,3
constant,3
binary,2
target,1


Once we set aside the three keys, the target and the three constant columns, 27 explanatory variables remain. Twelve are quantitative, nine are ordinal, four are nominal and two are binary. We treat `note_evaluation_actuelle` as ordinal, although it only takes the values 3 and 4. We assume that the five levels of `niveau_education` are ordered, since the brief does not describe them. We class `augementation_salaire_precedente` as quantitative because we will convert it from text to a number.

## 3. Data quality checks

### 3.1 Missing values

Missing values would force us to drop or impute rows, so we count them in each file.

In [7]:
pd.Series(
    {name: int(df.isna().sum().sum()) for name, df in sources.items()},
    name="n_missing",
).to_frame()

,n_missing
sirh,0
evaluations,0
survey,0


No file has any missing value, so we will not need any imputation.

### 3.2 Duplicated rows

Each file has a unique key, so two full rows can never be identical. We look instead for rows that are identical once the key is removed, which would reveal an employee entered twice.

In [8]:
JOIN_KEYS = {
    "sirh": "id_employee",
    "evaluations": "eval_number",
    "survey": "code_sondage",
}

pd.Series(
    {
        name: int(df.drop(columns=JOIN_KEYS[name]).duplicated().sum())
        for name, df in sources.items()
    },
    name="n_duplicated_rows",
).to_frame()

,n_duplicated_rows
sirh,0
evaluations,55
survey,16


`sirh` has no duplicate, while `evaluations` and `survey` have 55 and 16. These two files hold mostly scores from 1 to 5, so two employees can give the same answers by chance. Monthly income is nearly unique in `sirh`, so the absence of duplicates there shows that no employee appears twice.

### 3.3 Constant columns

A column with a single value cannot separate leavers from stayers, so we list the constant columns of each file.

In [9]:
constant_columns = {
    name: [col for col in df.columns if df[col].nunique(dropna=False) == 1]
    for name, df in sources.items()
}
constant_columns

{'sirh': ['nombre_heures_travailless'],
 'evaluations': [],
 'survey': ['nombre_employee_sous_responsabilite', 'ayant_enfants']}

#### Decision

We will drop the three constant columns after the join. Each one takes the same value for every employee, so it carries no information about attrition. The brief does not document what the value of 80 working hours stands for anyway.

### 3.4 Numeric-looking columns stored as text

A numeric column stored as text cannot enter the statistics or the model, so we look for text columns whose values are actually numbers. We check for the usual suspects: spaces, the percent sign, and the comma as decimal separator, since the files come from French systems.

In [10]:
rows = []
for source_name, df in sources.items():
    for col in df.select_dtypes(exclude="number").columns:
        values = df[col].dropna().astype(str)
        cleaned = (
            values.str.replace("%", "", regex=False)
            .str.replace(",", ".", regex=False)
            .str.strip()
        )
        rows.append(
            {
                "source": source_name,
                "column": col,
                "share_numeric": round(
                    pd.to_numeric(cleaned, errors="coerce").notna().mean(), 3
                ),
                "n_unique": values.nunique(),
                "examples": ", ".join(values.unique()[:3]),
            }
        )

pd.DataFrame(rows).sort_values("share_numeric", ascending=False)

,source,column,share_numeric,n_unique,examples
6,evaluations,augementation_salaire_precedente,1.0,15,"11 %, 23 %, 15 %"
1,sirh,statut_marital,0.0,3,"Célibataire, Marié(e), Divorcé(e)"
0,sirh,genre,0.0,2,"F, M"
2,sirh,departement,0.0,3,"Commercial, Consulting, Ressources Humaines"
3,sirh,poste,0.0,9,"Cadre Commercial, Assistant de Direction, Cons..."
4,evaluations,eval_number,0.0,1470,"E_1, E_2, E_4"
5,evaluations,heure_supplementaires,0.0,2,"Oui, Non"
7,survey,a_quitte_l_entreprise,0.0,2,"Oui, Non"
8,survey,domaine_etude,0.0,6,"Infra & Cloud, Autre, Transformation Digitale"
9,survey,ayant_enfants,0.0,1,Y


#### Decision

We will convert `augementation_salaire_precedente` to an integer by removing the percent sign. We keep the values in percentage points, such as 11 rather than 0.11, because they read more easily on charts. All values pass the numeric check, so the conversion will not create any missing value. We will apply it when we clean and join the three files.

### 3.5 Join keys

The three files use different formats for the employee key, so we normalize them before checking that they identify the same employees. `code_sondage` is written as `000001` in the file, but pandas already reads it as an integer.

In [11]:
keys = {name: to_employee_id(df[JOIN_KEYS[name]]) for name, df in sources.items()}

rows = []
for name, key in keys.items():
    others = [set(other) for other_name, other in keys.items() if other_name != name]
    rows.append(
        {
            "file": name,
            "n_rows": len(key),
            "n_distinct_keys": key.nunique(),
            "n_missing_elsewhere": len(set(key) - set.intersection(*others)),
        }
    )
display(pd.DataFrame(rows).set_index("file"))

assert all(key.is_unique for key in keys.values()), "a key is repeated in its file"
assert len({frozenset(key) for key in keys.values()}) == 1, "the key sets differ"

,n_rows,n_distinct_keys,n_missing_elsewhere
file,,,
sirh,1470,1470,0
evaluations,1470,1470,0
survey,1470,1470,0


#### Decision

Each key appears once in its file, and the three keys cover exactly the same 1,470 employees. An inner join and a left join therefore give the same result. We choose the inner join because the model needs employees present in all three files. The assertions above stop the notebook if the keys ever differ, so the join cannot lose an employee silently.